## 说明

请按照填空顺序编号分别完成参数优化，不同基函数的实现

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_data(filename):
    xys = []
    with open(filename, 'r') as f:
        for line in f:
            xys.append(map(float, line.strip().split()))
        xs, ys = zip(*xys)
        return np.asarray(xs), np.asarray(ys)


## 不同的基函数 (basis function) 填空顺序 2

请分别实现多项式基函数以及高斯基函数，训练集x的范围在0-25之间

In [ ]:
def identity_basis(x):
    ret = np.expand_dims(x, axis=1)
    return ret

def multinomial_basis(x, feature_num=10):
    """多项式基函数"""
    x = np.expand_dims(x, axis=1)  # shape (N, 1)
    # 从 x^1 到 x^feature_num，避免与外部偏置列重复
    ret = np.concatenate([x ** i for i in range(1, feature_num + 1)], axis=1)
    return ret

def gaussian_basis(x, feature_num=10):
    """高斯基函数"""
    x = np.expand_dims(x, axis=1)                   # shape (N, 1)
    centers = np.linspace(0, 25, feature_num)        # 在 0~25 均匀取中心
    sigma = (25.0 - 0.0) / (feature_num - 1)        # 相邻中心间距作为 sigma
    ret = np.exp(-(x - centers) ** 2 / (2 * sigma ** 2))
    return ret


## 最小二乘法 填空顺序 1

Normal Equation: w* = (Phi^T Phi)^{-1} Phi^T y

In [ ]:
def main(x_train, y_train):
    """
    训练模型（最小二乘法），返回从x到y的映射函数。
    """
    basis_func = gaussian_basis
    phi0 = np.expand_dims(np.ones_like(x_train), axis=1)
    phi1 = basis_func(x_train)
    phi = np.concatenate([phi0, phi1], axis=1)  # 设计矩阵 shape (N, D+1)

    # Normal Equation，用 pinv 避免奇异矩阵
    w = np.linalg.pinv(phi.T @ phi) @ phi.T @ y_train

    def f(x):
        phi0 = np.expand_dims(np.ones_like(x), axis=1)
        phi1 = basis_func(x)
        phi = np.concatenate([phi0, phi1], axis=1)
        return phi @ w

    return f


## 梯度下降法 填空顺序 3

L(w) = 1/(2N) ||Phi w - y||^2
grad = 1/N * Phi^T (Phi w - y)
w <- w - lr * grad

In [ ]:
def main_gd(x_train, y_train):
    """
    训练模型（梯度下降法），返回从x到y的映射函数。
    """
    basis_func = gaussian_basis
    phi0 = np.expand_dims(np.ones_like(x_train), axis=1)
    phi1 = basis_func(x_train)
    phi = np.concatenate([phi0, phi1], axis=1)

    N = phi.shape[0]
    w = np.zeros(phi.shape[1])
    lr = 0.1
    epochs = 2000

    for _ in range(epochs):
        residual = phi @ w - y_train
        gradient = phi.T @ residual / N
        w = w - lr * gradient

    def f(x):
        phi0 = np.expand_dims(np.ones_like(x), axis=1)
        phi1 = basis_func(x)
        phi = np.concatenate([phi0, phi1], axis=1)
        return phi @ w

    return f


## 评估结果

In [ ]:
def evaluate(ys, ys_pred):
    std = np.sqrt(np.mean(np.abs(ys - ys_pred) ** 2))
    return std

if __name__ == '__main__':
    x_train, y_train = load_data('train.txt')
    x_test, y_test = load_data('test.txt')
    print(x_train.shape, x_test.shape)

    f = main(x_train, y_train)
    print('[LS] train std: {:.4f}'.format(evaluate(y_train, f(x_train))))
    print('[LS] test  std: {:.4f}'.format(evaluate(y_test,  f(x_test))))

    f_gd = main_gd(x_train, y_train)
    print('[GD] train std: {:.4f}'.format(evaluate(y_train, f_gd(x_train))))
    print('[GD] test  std: {:.4f}'.format(evaluate(y_test,  f_gd(x_test))))

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(x_train, y_train, 'ro', markersize=3, label='train')
    plt.plot(x_test, f(x_test), 'k', label='LS pred')
    plt.title('Least Squares'); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(x_train, y_train, 'ro', markersize=3, label='train')
    plt.plot(x_test, f_gd(x_test), 'b', label='GD pred')
    plt.title('Gradient Descent'); plt.legend()

    plt.tight_layout()
    plt.show()
